# YOLO to ONNX Export and Validation

This notebook demonstrates how to export a YOLO model to ONNX format and validate its performance on a test set. The workflow includes:
- Exporting a trained YOLO model to ONNX
- Validating both the original YOLO and exported ONNX models on the same test set
- Comparing the evaluation metrics to ensure consistency and reliability

**Note:** This notebook follows best practices for code structure, naming conventions, and documentation.

In [ ]:
# Install required packages
%pip install onnx onnxruntime ultralytics opencv-python numpy

In [ ]:
from ultralytics import YOLO
import onnx
import onnxruntime as ort
import numpy as np
import cv2
import os
from typing import Tuple, Dict

# Set random seed for reproducibility
np.random.seed(42)

In [ ]:
class Config:
    """
    Configuration for model paths and evaluation settings.
    """
    # Path to the trained YOLO model weights
    MODEL_PATH = '/home/praktikan/projects/github/DwiAnggara/Automatic-Cutting-Description/notebooks/training/models/YOLO26m_Batch4_March_Dataset_1024_20260416_135745/weights/best.pt'
    # Path to the dataset YAML file
    DATA_YAML = '/home/praktikan/projects/DwiAnggara/Datasets/Batch3and4_YOLO/dataset.yaml'
    # Path to the test images directory
    TEST_IMAGES_DIR = '/home/praktikan/projects/DwiAnggara/Datasets/Batch3and4_YOLO/test/images'
    # Path to the ONNX export output
    ONNX_EXPORT_PATH = '/home/praktikan/projects/github/DwiAnggara/Automatic-Cutting-Description/notebooks/training/models/YOLO26m_Batch4_March_Dataset_1024_20260416_135745/weights/best.onnx'
    # Inference image size
    IMG_SIZE = 640
    # Confidence and IoU thresholds for evaluation
    CONF_THRESHOLD = 0.20
    IOU_THRESHOLD = 0.45
    # Maximum detections per image
    MAX_DET = 300
    # Use half precision (FP16) for YOLO inference
    HALF = False

config = Config()
print('Configuration initialized.')

In [ ]:
# 1. Load YOLO model and export to ONNX

def export_yolo_to_onnx(model_path: str, export_path: str, img_size: int = 640) -> str:
    """
    Export a YOLO model to ONNX format.
    Args:
        model_path (str): Path to YOLO weights (.pt).
        export_path (str): Output ONNX file path.
        img_size (int): Inference image size.
    Returns:
        str: Path to exported ONNX file.
    """
    model = YOLO(model_path)
    onnx_path = model.export(
        format="onnx",
        imgsz=img_size,
        half=False,
        dynamic=False,
        simplify=True,
        opset=17,
        nms=False,
        batch=1,
        device="cpu",
    )
    print(f"Exported ONNX model to: {onnx_path}")
    return onnx_path

onnx_path = export_yolo_to_onnx(config.MODEL_PATH, config.ONNX_EXPORT_PATH, config.IMG_SIZE)

In [ ]:
# 2. Validate the exported ONNX model

def validate_onnx_model(onnx_path: str) -> None:
    """
    Validate the exported ONNX model for correctness.
    Args:
        onnx_path (str): Path to the ONNX model file.
    """
    print("Validating ONNX model...")
    onnx_model = onnx.load(onnx_path)
    onnx.checker.check_model(onnx_model)
    print("ONNX model is valid.")

validate_onnx_model(onnx_path)

In [ ]:
# 3. Evaluate YOLO and ONNX models on the test set using YOLO Native Validation
import pandas as pd

def evaluate_model(model_path: str, cfg) -> dict:
    """
    Evaluate a model on the test dataset using the Ultralytics validation framework.
    Calculates key metrics like Precision, Recall, F1-Score, and mAP.
    This natively handles the standard YOLO dataset structure (images/labels routing).
    """
    print(f"Loading and validating model: {os.path.basename(model_path)}")
    
    # The Ultralytics wrapper intelligently infers the ONNX and PyTorch weights
    model = YOLO(model_path)
    
    # Ultralytics val handles datasets automatically mapping test splits
    metrics = model.val(
        data=cfg.DATA_YAML,
        split='test',
        conf=cfg.CONF_THRESHOLD,
        iou=cfg.IOU_THRESHOLD,
        imgsz=cfg.IMG_SIZE,
        max_det=cfg.MAX_DET,
        half=cfg.HALF,
        plots=False,
        save_json=False,
        verbose=False
    )
    
    # Extract bounding box metrics
    precision = metrics.box.mp
    recall = metrics.box.mr
    f1_score = 2 * (precision * recall) / (precision + recall + 1e-16)
    map50 = metrics.box.map50
    map50_95 = metrics.box.map
    
    # Extract computational latency per image mapping
    avg_latency_ms = metrics.speed['inference']
    
    return {
        'Model_Type': 'ONNX' if model_path.endswith('.onnx') else 'PyTorch (YOLO)',
        'Precision': precision,
        'Recall': recall,
        'F1_Score': f1_score,
        'mAP@50': map50,
        'mAP@50-95': map50_95,
        'Latency_ms_per_img': avg_latency_ms
    }

print("Running structural validation mapping on the test set...\n")

# 1. Base YOLO PyTorch Validation
yolo_metrics = evaluate_model(config.MODEL_PATH, config)

# 2. Exported ONNX Validation
onnx_metrics = evaluate_model(config.ONNX_EXPORT_PATH, config)

In [ ]:
# 4. Compare YOLO and ONNX results on the test set
from IPython.display import display

def compare_yolo_onnx_metrics(yolo_res: dict, onnx_res: dict) -> pd.DataFrame:
    """
    Consolidates and compares evaluation outputs into a structural matrix.
    """
    df = pd.DataFrame([yolo_res, onnx_res])
    
    # Determine the margin of variance between Native and Exported model
    variance = df.iloc[0].drop('Model_Type') - df.iloc[1].drop('Model_Type')
    variance['Model_Type'] = 'Variance (PyTorch - ONNX)'
    
    # Concolidate for visual reporting
    df = pd.concat([df, pd.DataFrame([variance])], ignore_index=True)
    
    # Restructure styling index gracefully
    df.set_index('Model_Type', inplace=True)
    return df

print("\n📊 YOLO vs ONNX Inference Validation Results:")
comparison_df = compare_yolo_onnx_metrics(yolo_metrics, onnx_metrics)

# Standard visual table display (native formatting in IPYNB format)
display(comparison_df)